In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import BertTokenizer, BertModel, AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Cell 2: Define Dataset Class
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

# Cell 3: Define Model Class
class BengaliTextClassifier(nn.Module):
    def __init__(self, n_classes):
        super(BengaliTextClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('sagorsarker/bangla-bert-base')
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.pooler_output
        dropout_output = self.dropout(pooled_output)
        logits = self.classifier(dropout_output)
        return self.sigmoid(logits)

# Cell 4: Define Training Function
def train_model(model, train_loader, val_loader, device, epochs=5):
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.BCELoss()
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        # Validation
        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f'Epoch {epoch + 1}:')
        print(f'Average training loss: {avg_train_loss:.4f}')
        print(f'Average validation loss: {avg_val_loss:.4f}')

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

# Cell 5: Data Loading and Preprocessing
# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/Thesis/ThesisDataSet.xlsx')

# Define label columns
label_columns = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']
texts = df['Abuse Data Text'].values
labels = df[label_columns].values

# Split the data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Cell 6: Model Setup and Training
# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('sagorsarker/bangla-bert-base')

# Create datasets
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BengaliTextClassifier(n_classes=len(label_columns))
model.to(device)

# Train the model
train_model(model, train_loader, val_loader, device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/2.24M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

...

Epoch 3:
Average training loss: 0.0822
Average validation loss: 0.1297


Epoch 4/5: 100%|██████████| 509/509 [03:06<00:00,  2.73it/s]


Epoch 4:
Average training loss: 0.0676
Average validation loss: 0.1265


Epoch 5/5: 100%|██████████| 509/509 [03:06<00:00,  2.73it/s]


Epoch 5:
Average training loss: 0.0557
Average validation loss: 0.1317


In [ ]:
def predict_text(text, model, tokenizer, device, label_columns):
    """
    Predict classification labels for a given Bengali text.

    Args:
        text (str): Input Bengali text
        model: Trained BengaliTextClassifier model
        tokenizer: BERT tokenizer
        device: torch device
        label_columns (list): List of label names

    Returns:
        dict: Dictionary containing predictions for each label
    """
    # Set model to evaluation mode
    model.eval()

    # Tokenize the input text
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    # Move inputs to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get predictions
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)

    # Convert predictions to probabilities
    predictions = outputs.cpu().numpy()[0]

    # Create dictionary of predictions
    results = {}
    for label, prob in zip(label_columns, predictions):
        results[label] = float(prob)

    return results

# Example usage
if __name__ == "__main__":
    # Load the saved model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    label_columns = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']

    # Initialize tokenizer and model
    tokenizer = BertTokenizer.from_pretrained('sagorsarker/bangla-bert-base')
    model = BengaliTextClassifier(n_classes=len(label_columns))

    # Load the saved model weights
    model.load_state_dict(torch.load('best_model.pt', map_location=device))
    model.to(device)

    # Example text for prediction
    sample_text = "চুদিরভাই"

    # Get predictions
    predictions = predict_text(sample_text, model, tokenizer, device, label_columns)

    # Print results
    print("\nPrediction Results:")
    print("-" * 50)
    for label, probability in predictions.items():
        print(f"{label}: {probability:.4f} ({probability*100:.2f}%)")

    # Print the most likely categories (probability > 0.5)
    print("\nDetected Categories:")
    print("-" * 50)
    detected = [label for label, prob in predictions.items() if prob > 0.5]
    if detected:
        print("This text contains:", ", ".join(detected))
    else:
        print("No categories detected (all probabilities below 50%)")

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
<ipython-input-6-7dc3f511951b>:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unles


Prediction Results:
--------------------------------------------------
Toxic: 0.9971 (99.71%)
Threat: 0.0049 (0.49%)
Obscene: 0.9846 (98.46%)
Insult: 0.9607 (96.07%)
Racism: 0.0078 (0.78%)

Detected Categories:
--------------------------------------------------
This text contains: Toxic, Obscene, Insult


In [6]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# FINAL

In [3]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import BertTokenizer, BertModel, AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

class TextClassificationDataset(Dataset):
    def __init__(self, texts, aspects, labels, tokenizer, max_length=128):
        self.texts = texts
        self.aspects = aspects
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Combine text and aspect
        combined_text = f"{str(self.texts[idx])} [SEP] {str(self.aspects[idx])}"
        label = self.labels[idx]

        encoding = self.tokenizer(
            combined_text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

class BengaliTextClassifier(nn.Module):
    def __init__(self, n_classes):
        super(BengaliTextClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('sagorsarker/bangla-bert-base')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.pooler_output
        dropout_output = self.dropout(pooled_output)
        logits = self.classifier(dropout_output)
        return self.sigmoid(logits)

def train_model(model, train_loader, val_loader, device, epochs=5):
    optimizer = AdamW(model.parameters(), lr=2e-5)
    criterion = nn.BCELoss()
    best_val_loss = float('inf')

    all_val_predictions = []
    all_val_labels = []

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        # Validation
        model.eval()
        total_val_loss = 0
        epoch_predictions = []
        epoch_labels = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']

                outputs = model(input_ids, attention_mask).cpu()
                val_predictions = (outputs > 0.5).float()

                epoch_predictions.append(val_predictions)
                epoch_labels.append(labels)

                loss = criterion(outputs, labels.float())
                total_val_loss += loss.item()

        # Concatenate predictions and labels
        epoch_predictions = torch.cat(epoch_predictions)
        epoch_labels = torch.cat(epoch_labels)

        # Metrics
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f'Epoch {epoch + 1}:')
        print(f'Average training loss: {avg_train_loss:.4f}')
        print(f'Average validation loss: {avg_val_loss:.4f}')

        # Store predictions for final evaluation
        all_val_predictions.append(epoch_predictions)
        all_val_labels.append(epoch_labels)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pt')

    # Final evaluation
    final_predictions = torch.cat(all_val_predictions)
    final_labels = torch.cat(all_val_labels)

    # Detailed Metrics
    label_names = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']
    print("\nDetailed Classification Report:")
    print(classification_report(
        final_labels.numpy(),
        final_predictions.numpy(),
        target_names=label_names
    ))

    # Confusion Matrix
    cm = confusion_matrix(
        final_labels.numpy().argmax(axis=1),
        final_predictions.numpy().argmax(axis=1)
    )
    plt.figure(figsize=(10,7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names,
                yticklabels=label_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()

#main
df = pd.read_excel('/content/drive/MyDrive/Thesis/version50k.xlsx')

# Define label columns
label_columns = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']

# Check if the label columns exist in the DataFrame
for col in label_columns:
    if col not in df.columns:
        print(f"Warning: Column '{col}' not found in DataFrame. Check your data.")

# Prepare data
texts = df['Abuse Data Text'].values
aspects = df['Aspects'].values

labels = df.loc[:, label_columns].values  # Updated using .loc

# Split the data
train_texts, val_texts, train_aspects, val_aspects, train_labels, val_labels = train_test_split(
    texts, aspects, labels, test_size=0.2, random_state=42
)

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('sagorsarker/bangla-bert-base')

# Create datasets
train_dataset = TextClassificationDataset(train_texts, train_aspects, train_labels, tokenizer)
val_dataset = TextClassificationDataset(val_texts, val_aspects, val_labels, tokenizer)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BengaliTextClassifier(n_classes=len(label_columns))
model.to(device)

# Train the model
train_model(model, train_loader, val_loader, device)

KeyError: "None of [Index(['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism'], dtype='object')] are in the [columns]"

In [3]:
!pip install numpy==1.25.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 40.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.0
    Uninstalling numpy-2.2.0:
      Successfully uninstalled numpy-2.2.0


In [1]:
!pip install pandas==1.5.3  # Replace with the compatible version
!pip install transformers==4.28.1  # Replace with the compatible version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 56.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.10.1 requires pandas<2.2.3dev0,>=2.0, but you have pandas 1.5.3 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 1.5.3 which is incompatible.
mizani 0.13.1 requires pandas>=2.2.0, but you have pandas 1.5.3 which is incompatible.
plotnine 0.14.3 requires pandas>=2.2.0, but you have pandas 1.5.3 which is incompatible.
xarray 2024.10.0 requires pandas>=2.1, but you have pandas 1.5.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 6

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm

def evaluate_model(model, test_loader, device, threshold=0.5):
    """
    Evaluate the model's performance using various metrics.

    Args:
        model: The trained model
        test_loader: DataLoader containing test data
        device: Device to run the evaluation on
        threshold: Classification threshold (default: 0.5)

    Returns:
        dict: Dictionary containing various performance metrics
    """
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].cpu().numpy()

            outputs = model(input_ids, attention_mask)
            predictions = (outputs.cpu().numpy() > threshold).astype(int)

            all_predictions.extend(predictions)
            all_labels.extend(labels)

    # Convert lists to numpy arrays
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)

    # Calculate metrics for each class
    results = {}
    label_names = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']

    # Overall accuracy (sample-wise)
    sample_accuracy = np.mean(np.all(all_predictions == all_labels, axis=1))
    results['overall_sample_accuracy'] = sample_accuracy

    # Per-class metrics
    for i, label in enumerate(label_names):
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels[:, i],
            all_predictions[:, i],
            average='binary'
        )
        accuracy = accuracy_score(all_labels[:, i], all_predictions[:, i])

        results[f'{label}_accuracy'] = accuracy
        results[f'{label}_precision'] = precision
        results[f'{label}_recall'] = recall
        results[f'{label}_f1'] = f1

    # Calculate macro-averaged metrics
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average='macro'
    )

    results['macro_precision'] = macro_precision
    results['macro_recall'] = macro_recall
    results['macro_f1'] = macro_f1

    return results

def print_evaluation_results(results):
    """
    Print evaluation results in a formatted way.

    Args:
        results: Dictionary containing the evaluation metrics
    """
    print("\n=== Model Evaluation Results ===")
    print(f"\nOverall Sample-wise Accuracy: {results['overall_sample_accuracy']:.4f}")

    print("\n=== Per-Class Metrics ===")
    label_names = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']

    for label in label_names:
        print(f"\n{label}:")
        print(f"Accuracy:  {results[f'{label}_accuracy']:.4f}")
        print(f"Precision: {results[f'{label}_precision']:.4f}")
        print(f"Recall:    {results[f'{label}_recall']:.4f}")
        print(f"F1-Score:  {results[f'{label}_f1']:.4f}")

    print("\n=== Macro-Averaged Metrics ===")
    print(f"Macro Precision: {results['macro_precision']:.4f}")
    print(f"Macro Recall:    {results['macro_recall']:.4f}")
    print(f"Macro F1-Score:  {results['macro_f1']:.4f}")

# Example usage:
# Load the best model
model.load_state_dict(torch.load('best_model.pt'))
model.to(device)

# Create test dataset and loader (using validation data for this example)
test_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

# Evaluate the model
evaluation_results = evaluate_model(model, test_loader, device)
print_evaluation_results(evaluation_results)

# optimized

## LLM

In [ ]:
import torch
from torch import nn
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Pipeline,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import pandas as pd
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:


class BengaliLLMClassifier:
    def __init__(self, model_name="csebuetnlp/banglabert", num_labels=5):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type="multi_label_classification"
        )

    def prepare_data(self, df, text_column, label_columns):
        # Ensure text column contains strings
        texts = df[text_column].astype(str).tolist()

        # Convert labels to numpy array and ensure they're floats
        labels = df[label_columns].values.astype(np.float32).tolist()

        # Create dataset dictionary with proper types
        dataset_dict = {
            'text': texts,
            'labels': labels
        }

        # Create dataset
        dataset = Dataset.from_dict(dataset_dict)

        # Tokenize function
        def tokenize_function(examples):
            return self.tokenizer(
                examples['text'],
                padding='max_length',
                truncation=True,
                max_length=512
            )

        # Tokenize dataset
        tokenized_dataset = dataset.map(tokenize_function, batched=True)

        # Set format for pytorch
        tokenized_dataset.set_format(
            type='torch',
            columns=['input_ids', 'attention_mask', 'labels']
        )

        return tokenized_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = (predictions > 0.5).astype(int)

        # Calculate metrics
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels,
            predictions,
            average='weighted',
            zero_division=0
        )

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    def train(self, train_dataset, val_dataset, output_dir='./results'):
        training_args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=2e-5,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            num_train_epochs=3,
            weight_decay=0.01,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            push_to_hub=False,
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics
        )

        trainer.train()
        return trainer

class MultilingualLLMClassifier(BengaliLLMClassifier):
    def __init__(self, num_labels=5):
        super().__init__(
            model_name="xlm-roberta-large",
            num_labels=num_labels
        )

class InstructionTunedClassifier(BengaliLLMClassifier):
    def __init__(self, model_name="google/flan-t5-large"):
        super().__init__(model_name=model_name, num_labels=5)

    def prepare_instruction_data(self, df, text_column, label_columns):
        # Create instructions and ensure proper data types
        instructions = []
        for text in df[text_column]:
            instruction = (
                "Classify the following Bengali text into categories: "
                "Toxic, Threat, Obscene, Insult, and Racism. "
                f"Text: {str(text)}"
            )
            instructions.append(instruction)

        # Convert labels to numpy array and ensure they're floats
        labels = df[label_columns].values.astype(np.float32).tolist()

        dataset_dict = {
            'text': instructions,
            'labels': labels
        }

        return Dataset.from_dict(dataset_dict)

def main():
    # Load your data
    df = pd.read_excel('/content/drive/MyDrive/ResarchColab/ThesisDataSet.xlsx')

    # Define columns
    text_column = 'Abuse Data Text'
    label_columns = ['Toxic', 'Threat', 'Obscene', 'Insult', 'Racism']

    # Ensure label columns are float type
    for col in label_columns:
        df[col] = df[col].astype(float)

    # Split data
    train_df = df.sample(frac=0.8, random_state=42)
    val_df = df.drop(train_df.index)

    try:
        # 1. Bengali-specialized LLM
        print("Training Bengali-specialized LLM...")
        bengali_classifier = BengaliLLMClassifier()
        train_dataset = bengali_classifier.prepare_data(train_df, text_column, label_columns)
        val_dataset = bengali_classifier.prepare_data(val_df, text_column, label_columns)
        trainer = bengali_classifier.train(train_dataset, val_dataset)

        # 2. Multilingual LLM
        print("\nTraining Multilingual LLM...")
        multi_classifier = MultilingualLLMClassifier()
        train_dataset = multi_classifier.prepare_data(train_df, text_column, label_columns)
        val_dataset = multi_classifier.prepare_data(val_df, text_column, label_columns)
        trainer = multi_classifier.train(train_dataset, val_dataset)

        # 3. Instruction-tuned LLM
        print("\nTraining Instruction-tuned LLM...")
        instruction_classifier = InstructionTunedClassifier()
        train_dataset = instruction_classifier.prepare_instruction_data(
            train_df, text_column, label_columns
        )
        val_dataset = instruction_classifier.prepare_instruction_data(
            val_df, text_column, label_columns
        )
        trainer = instruction_classifier.train(train_dataset, val_dataset)

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        raise

if __name__ == "__main__":
    main()